# Phase 2 – L1 Feature Selection via Logistic Regression

Uses `LogisticRegression(penalty='l1', solver='saga')` as the selector inside `SelectFromModel`.

**Why LogisticRegression instead of LinearSVC:**
- `solver='saga'` is an incremental gradient solver — handles large feature spaces (265 features) without convergence issues
- No `ConvergenceWarning` spam
- Equivalent L1 sparsity to LinearSVC but more numerically stable

**Resilience features:**
- Results saved per-dataset to `phase2_results/dataset_XX.json` immediately after each dataset
- Re-running skips already-completed datasets automatically
- Per-fold timing printed so you can monitor progress

## 1. Imports

In [1]:
import csv
import json
import time
import warnings
from pathlib import Path

import numpy as np
from openpyxl import load_workbook

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print('All imports OK')

All imports OK


## 2. Paths & Constants

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'data' / 'processed' / 'Datasets').exists():
            return candidate
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT  = find_project_root(Path.cwd())
DATASETS_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'Datasets'
RESULTS_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'phase2_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = 'Label'
RANDOM_STATE  = 42

# C grid for L1 LogisticRegression
# Small C = strong regularization = fewer features selected
# Large C = weak regularization  = more features kept
L1_C_GRID = [0.001, 0.01, 0.1, 1.0]

print('PROJECT_ROOT :', PROJECT_ROOT)
print('RESULTS_DIR  :', RESULTS_DIR)
print('L1_C_GRID    :', L1_C_GRID)

PROJECT_ROOT : /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection
RESULTS_DIR  : /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection/data/processed/phase2_results
L1_C_GRID    : [0.001, 0.01, 0.1, 1.0]


## 3. Classifiers & Outer CV

In [3]:
classifiers = {
    'SVM':          SVC(random_state=RANDOM_STATE),
    'kNN':          KNeighborsClassifier(),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE),
    'MLP':          MLPClassifier(random_state=RANDOM_STATE, max_iter=500),
}

outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

print('Classifiers:', list(classifiers.keys()))
print('Outer CV   :', outer_cv)

Classifiers: ['SVM', 'kNN', 'DecisionTree', 'RandomForest', 'MLP']
Outer CV   : StratifiedKFold(n_splits=10, random_state=42, shuffle=True)


## 4. Data Loader

In [4]:
def load_train_dataset(dataset_id):
    train_csv = DATASETS_ROOT / str(dataset_id) / 'train.csv'
    if not train_csv.exists():
        raise FileNotFoundError(f'Missing: {train_csv}')
    with train_csv.open(newline='') as f:
        reader = csv.DictReader(f)
        feature_names = [c for c in reader.fieldnames if c != TARGET_COLUMN]
        rows = list(reader)

    def to_float(v):
        if v is None: return np.nan
        v = v.strip()
        return np.nan if v == '' else float(v)

    X = np.array([[to_float(r[c]) for c in feature_names] for r in rows], dtype=float)
    y = np.array([r[TARGET_COLUMN] for r in rows])
    return X, y, feature_names

## 5. L1 LogisticRegression Selector & Pipeline Builder

**C selection strategy:** Single stratified 80/20 split on `X_train` (first fold of a 5-fold splitter).  
This is ~5x faster than full inner CV and gives equivalent C selection quality.  
Imputation and scaling are done once and shared across all C values for efficiency.

In [5]:
def make_l1_logreg(C):
    """LogisticRegression with L1 penalty — the feature selector estimator."""
    return LogisticRegression(
        C=C,
        penalty='l1',
        solver='saga',       # only solver supporting L1 + multiclass
        max_iter=2000,
        tol=1e-2,            # loose tolerance: fine for feature selection
        random_state=RANDOM_STATE,
        n_jobs=-1,           # use all CPU cores
    )


def select_best_l1_C(X_train, y_train, c_grid=L1_C_GRID):
    """
    Choose the best L1 C via a single fast 80/20 split on X_train.
    Pre-fits imputer + scaler once and reuses across all C values.
    Falls back to c_grid[1] (0.01) if all C values zero out all features.
    """
    # Single stratified split (first fold of 5-fold = ~80/20)
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    tr_idx, val_idx = next(splitter.split(X_train, y_train))
    Xtr,  Xval  = X_train[tr_idx],  X_train[val_idx]
    ytr,  yval  = y_train[tr_idx],  y_train[val_idx]

    # Pre-process once (shared across all C values)
    imputer  = SimpleImputer(strategy='median').fit(Xtr)
    Xtr_imp  = imputer.transform(Xtr)
    Xval_imp = imputer.transform(Xval)
    scaler   = StandardScaler().fit(Xtr_imp)
    Xtr_sc   = scaler.transform(Xtr_imp)
    Xval_sc  = scaler.transform(Xval_imp)

    best_C, best_score = c_grid[1], -1.0  # safe default

    for C in c_grid:
        try:
            # Fit selector
            selector = SelectFromModel(make_l1_logreg(C), prefit=False)
            selector.fit(Xtr_sc, ytr)

            mask = selector.get_support()
            if mask.sum() == 0:
                continue  # C too small — zeroed everything out

            # Quick probe score using a fast LR on the reduced set
            probe = LogisticRegression(
                C=1.0, solver='saga', max_iter=500,
                tol=1e-2, random_state=RANDOM_STATE, n_jobs=-1
            )
            probe.fit(Xtr_sc[:, mask], ytr)
            score = accuracy_score(yval, probe.predict(Xval_sc[:, mask]))

            if score > best_score:
                best_score, best_C = score, C

        except Exception:
            continue  # skip this C if anything goes wrong

    return best_C


def build_l1_pipeline(clf_name, clf_model, l1_C):
    """
    Full pipeline:
      Imputer -> Scaler -> SelectFromModel(LogReg L1) -> [Scaler2] -> Classifier

    The second scaler (scaler2) is only added for distance/gradient-sensitive
    classifiers (SVM, kNN, MLP) to re-standardise after the feature mask is applied.
    """
    steps = [
        ('imputer',  SimpleImputer(strategy='median')),
        ('scaler',   StandardScaler()),
        ('selector', SelectFromModel(make_l1_logreg(l1_C), prefit=False)),
    ]

    if clf_name in ['SVM', 'kNN', 'MLP']:
        steps.append(('scaler2', StandardScaler()))

    steps.append(('clf', clone(clf_model)))
    return Pipeline(steps)


print('Selector and pipeline builder defined.')
print('Selector estimator: LogisticRegression(penalty=l1, solver=saga)')

Selector and pipeline builder defined.
Selector estimator: LogisticRegression(penalty=l1, solver=saga)


## 6. Per-Dataset Save / Load Helpers

In [6]:
def result_path(dataset_id):
    return RESULTS_DIR / f'dataset_{dataset_id:02d}.json'

def save_dataset_result(dataset_id, data):
    with result_path(dataset_id).open('w') as f:
        json.dump(data, f, indent=2)
    print(f'  Saved → {result_path(dataset_id).name}')

def load_dataset_result(dataset_id):
    p = result_path(dataset_id)
    if not p.exists():
        return None
    with p.open() as f:
        return json.load(f)

def is_done(dataset_id):
    return result_path(dataset_id).exists()

# Show current progress
done = [i for i in range(1, 17) if is_done(i)]
todo = [i for i in range(1, 17) if not is_done(i)]
print(f'Already completed : {done if done else "none"}')
print(f'Still to run      : {todo if todo else "none — all done!"}')

Already completed : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Still to run      : [12, 13, 14, 15, 16]


## 7. Main Experiment Loop

- Skips datasets that already have a saved JSON
- Saves immediately after each dataset completes
- Prints: classifier, fold, accuracy, macro-F1, features selected/total, best C, elapsed seconds

In [7]:
for dataset_id in range(1, 17):

    # ── Skip completed datasets ────────────────────────────────────────────
    if is_done(dataset_id):
        print(f'Dataset {dataset_id:2d} — already done, skipping.')
        continue

    t_ds = time.time()
    X, y, feature_names = load_train_dataset(dataset_id)
    feature_names_arr = np.array(feature_names)
    n_features = len(feature_names)

    print(f'\n=== Dataset {dataset_id} | shape={X.shape} | features={n_features} ===')

    dataset_result = {}

    for clf_name, clf_model in classifiers.items():
        fold_rows = []
        t_clf = time.time()
        print(f'  --- {clf_name} ---')

        for fold_idx, (train_idx, valid_idx) in enumerate(
                outer_cv.split(X, y), start=1):

            t_fold = time.time()
            X_train, X_valid = X[train_idx], X[valid_idx]
            y_train, y_valid = y[train_idx], y[valid_idx]

            # 1. Select best C via fast single-split inner evaluation
            best_C = select_best_l1_C(X_train, y_train)

            # 2. Build and fit the full pipeline on X_train only
            pipeline = build_l1_pipeline(clf_name, clf_model, best_C)
            pipeline.fit(X_train, y_train)

            # 3. Evaluate on validation fold
            y_pred = pipeline.predict(X_valid)
            acc    = accuracy_score(y_valid, y_pred)
            mf1    = f1_score(y_valid, y_pred, average='macro', zero_division=0)

            # 4. Extract selected features
            selector      = pipeline.named_steps['selector']
            mask          = selector.get_support()
            selected      = feature_names_arr[mask].tolist()
            n_selected    = int(mask.sum())

            fold_rows.append({
                'fold':              fold_idx,
                'accuracy':          round(acc, 6),
                'macro_f1':          round(mf1, 6),
                'n_selected':        n_selected,
                'selected_features': selected,
                'best_l1_C':         best_C,
            })

            elapsed = time.time() - t_fold
            print(f'    fold {fold_idx:2d} | '
                  f'acc={acc:.4f} | f1={mf1:.4f} | '
                  f'features={n_selected:3d}/{n_features} | '
                  f'C={best_C} | {elapsed:.1f}s')

        dataset_result[clf_name] = fold_rows

        # Per-classifier summary
        acc_mean = np.mean([r['accuracy']   for r in fold_rows])
        acc_std  = np.std( [r['accuracy']   for r in fold_rows])
        f1_mean  = np.mean([r['macro_f1']   for r in fold_rows])
        f1_std   = np.std( [r['macro_f1']   for r in fold_rows])
        n_mean   = np.mean([r['n_selected'] for r in fold_rows])
        clf_time = time.time() - t_clf
        print(f'  => {clf_name:12s} | '
              f'acc={acc_mean:.4f}±{acc_std:.4f} | '
              f'f1={f1_mean:.4f}±{f1_std:.4f} | '
              f'avg_features={n_mean:.1f}/{n_features} | '
              f'total={clf_time:.0f}s\n')

    # ── Save this dataset immediately ──────────────────────────────────────
    save_dataset_result(dataset_id, dataset_result)
    print(f'Dataset {dataset_id} complete in {time.time() - t_ds:.0f}s')

print('\nAll datasets done.')

Dataset  1 — already done, skipping.
Dataset  2 — already done, skipping.
Dataset  3 — already done, skipping.
Dataset  4 — already done, skipping.
Dataset  5 — already done, skipping.
Dataset  6 — already done, skipping.
Dataset  7 — already done, skipping.
Dataset  8 — already done, skipping.
Dataset  9 — already done, skipping.
Dataset 10 — already done, skipping.
Dataset 11 — already done, skipping.

=== Dataset 12 | shape=(8330, 265) | features=265 ===
  --- SVM ---
    fold  1 | acc=0.8079 | f1=0.0705 | features=  2/265 | C=0.001 | 30.7s
    fold  2 | acc=0.8127 | f1=0.0830 | features= 95/265 | C=0.01 | 32.6s
    fold  3 | acc=0.8127 | f1=0.0937 | features= 75/265 | C=0.01 | 39.0s
    fold  4 | acc=0.8115 | f1=0.1025 | features=232/265 | C=0.1 | 25.0s
    fold  5 | acc=0.8115 | f1=0.0827 | features=  4/265 | C=0.001 | 21.5s
    fold  6 | acc=0.8139 | f1=0.0709 | features=  2/265 | C=0.001 | 23.1s
    fold  7 | acc=0.8091 | f1=0.0680 | features=  4/265 | C=0.001 | 23.5s
    fold  

## 8. Merge All Per-Dataset Files → Single JSON

Run after all 16 datasets are complete.

In [8]:
l1_results = {}
missing    = []

for dataset_id in range(1, 17):
    data = load_dataset_result(dataset_id)
    if data is None:
        missing.append(dataset_id)
        print(f'  WARNING: Dataset {dataset_id} not found')
    else:
        l1_results[dataset_id] = data

if missing:
    print(f'\nMissing: {missing}. Run Cell 7 again to complete them.')
else:
    merged_path = PROJECT_ROOT / 'data' / 'processed' / 'l1_results.json'
    with merged_path.open('w') as f:
        json.dump(l1_results, f, indent=2)
    print(f'Merged {len(l1_results)} datasets → {merged_path}')

Merged 16 datasets → /Users/sultanarazia/Desktop/UNB/Machine Learning/Project/L1-feature-selection/data/processed/l1_results.json


## 9. Export to Excel — "After FS-DR" Sheet

In [ ]:
if missing:
    print('Cannot export — complete all datasets first.')
else:
    results_xlsx = DATASETS_ROOT / 'Results.xlsx'
    wb = load_workbook(results_xlsx)
    ws = wb['After FS-DR']

    column_map = {
        'SVM':          ('B', 'C', 'L'),
        'kNN':          ('D', 'E', 'M'),
        'DecisionTree': ('F', 'G', 'N'),
        'RandomForest': ('H', 'I', 'O'),
        'MLP':          ('J', 'K', 'P'),
    }

    for dataset_id in range(1, 17):
        start_row = 3 + (dataset_id - 1) * 12
        for clf_name, rows in l1_results[dataset_id].items():
            acc_col, f1_col, param_col = column_map[clf_name]
            for row in rows:
                excel_row = start_row + row['fold'] - 1
                ws[f'{acc_col}{excel_row}'] = row['accuracy']
                ws[f'{f1_col}{excel_row}']  = row['macro_f1']
                ws[f'{param_col}{excel_row}'] = str({
                    'selector':            'LogisticRegression(L1, saga)',
                    'l1_C':                row['best_l1_C'],
                    'n_features_selected': row['n_selected'],
                    'selected_features':   row['selected_features'],
                })

    wb.save(results_xlsx)
    print(f'Exported to: {results_xlsx}')

## 10. Summary Table (mean ± std)

In [9]:
if not l1_results:
    print('No results loaded yet — run Cells 7 and 8 first.')
else:
    print(f"{'DS':<4} {'Classifier':<14} {'Accuracy':>20} {'Macro-F1':>20} {'Avg Features':>14}")
    print('-' * 76)
    prev_ds = None
    for dataset_id in sorted(l1_results.keys(), key=int):
        if prev_ds and prev_ds != dataset_id:
            print()
        for clf_name, rows in l1_results[dataset_id].items():
            acc = [r['accuracy']   for r in rows]
            f1  = [r['macro_f1']   for r in rows]
            n   = [r['n_selected'] for r in rows]
            print(f'{dataset_id:<4} {clf_name:<14} '
                  f'{np.mean(acc):.4f} ± {np.std(acc):.4f}   '
                  f'{np.mean(f1):.4f} ± {np.std(f1):.4f}   '
                  f'{np.mean(n):.1f}')
        prev_ds = dataset_id

DS   Classifier                 Accuracy             Macro-F1   Avg Features
----------------------------------------------------------------------------
1    SVM            0.9520 ± 0.0074   0.7806 ± 0.1157   12.0
1    kNN            0.9784 ± 0.0049   0.8133 ± 0.1183   12.0
1    DecisionTree   1.0000 ± 0.0000   1.0000 ± 0.0000   12.0
1    RandomForest   0.9999 ± 0.0003   0.9999 ± 0.0003   12.0
1    MLP            0.9861 ± 0.0036   0.9563 ± 0.0734   12.0

2    SVM            0.9310 ± 0.0072   0.4374 ± 0.0097   11.7
2    kNN            0.9520 ± 0.0076   0.5563 ± 0.0596   11.7
2    DecisionTree   0.9585 ± 0.0057   0.5991 ± 0.0482   11.7
2    RandomForest   0.9601 ± 0.0067   0.5906 ± 0.0554   11.7
2    MLP            0.9514 ± 0.0067   0.5167 ± 0.0515   11.7

3    SVM            0.8713 ± 0.0060   0.4698 ± 0.0272   12.0
3    kNN            0.8898 ± 0.0071   0.5811 ± 0.0166   12.0
3    DecisionTree   0.8898 ± 0.0066   0.5801 ± 0.0257   12.0
3    RandomForest   0.8950 ± 0.0076   0.5938 ± 0.03

## 11. Inspect a Single Dataset (optional)

Check any individual dataset result at any time — useful while the main loop is still running.

In [10]:
INSPECT_DATASET = 9   # change to any id 1–16

data = load_dataset_result(INSPECT_DATASET)
if data is None:
    print(f'Dataset {INSPECT_DATASET} not yet completed.')
else:
    print(f'Dataset {INSPECT_DATASET} — Phase 2 results:')
    print(f"  {'Classifier':<14} {'Accuracy':>18} {'Macro-F1':>18} {'Avg Features':>14}")
    print('  ' + '-' * 66)
    for clf_name, rows in data.items():
        acc = [r['accuracy']   for r in rows]
        f1  = [r['macro_f1']   for r in rows]
        n   = [r['n_selected'] for r in rows]
        print(f"  {clf_name:<14} "
              f"{np.mean(acc):.4f} ± {np.std(acc):.4f}   "
              f"{np.mean(f1):.4f} ± {np.std(f1):.4f}   "
              f"{np.mean(n):.1f}")

    # Show which features were most consistently selected across all folds x classifiers
    from collections import Counter
    all_features = []
    for rows in data.values():
        for r in rows:
            all_features.extend(r['selected_features'])
    top = Counter(all_features).most_common(10)
    total_selections = len(list(data.values())[0]) * len(data)  # folds x classifiers
    print(f'\n  Top 10 most consistently selected features (out of {total_selections} total fold×clf combos):')
    for feat, count in top:
        print(f'    {feat:<30} selected {count:3d} times')

Dataset 9 — Phase 2 results:
  Classifier               Accuracy           Macro-F1   Avg Features
  ------------------------------------------------------------------
  SVM            0.9503 ± 0.0065   0.7271 ± 0.0453   115.2
  kNN            0.9529 ± 0.0064   0.7761 ± 0.0485   115.2
  DecisionTree   0.9822 ± 0.0043   0.8584 ± 0.0366   115.2
  RandomForest   0.9910 ± 0.0024   0.9243 ± 0.0264   115.2
  MLP            0.9783 ± 0.0043   0.8855 ± 0.0256   115.2

  Top 10 most consistently selected features (out of 50 total fold×clf combos):
    F1                             selected  50 times
    F2                             selected  50 times
    F4                             selected  50 times
    F6                             selected  50 times
    F7                             selected  50 times
    F9                             selected  50 times
    F11                            selected  50 times
    F13                            selected  50 times
    F14                 